# NB-10-01 — MDPs and Dynamic Programming

**Module M10 · Reinforcement Learning** · [L10-01](../lessons/L10-01-foundations-of-reinforcement-learning.md) ·
[L10-02](../lessons/L10-02-value-based-reinforcement-learning.md)

Build an MDP in code, compute returns, solve it exactly, and see how the discount changes both the values
and the policy. Nothing here needs a model-free algorithm: the point is the reference solution every later
notebook is compared against.

In [1]:
import sys
sys.path.insert(0, "../code")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rl_envs, dp, tabular, experiments

print("experiments self-test:", experiments.run_tests())

experiments self-test: True


## 1. The environment

`GridMDP` is a small deterministic gridworld with a start, a goal and optional slip. States and actions are
integers; transitions and rewards are known, which is what "has a model" means.

In [2]:
mdp = rl_envs.make_gridworld("open", step_cost=-0.04)
print("states:", mdp.n_states, "actions:", mdp.n_actions)
print(dp.format_grid(mdp, np.zeros(mdp.n_states)))

env = rl_envs.MDPAdapter(mdp)
obs, info = env.reset(seed=0)
print("reset:", obs, info)
obs, reward, terminated, truncated, info = env.step(0)
print("one step:", obs, reward, terminated, truncated, info)

states: 25 actions: 4
+0.00 +0.00 +0.00 +0.00 +0.00
+0.00 +0.00 +0.00 +0.00 +0.00
+0.00 +0.00 +0.00 +0.00 +0.00
+0.00 +0.00 +0.00 +0.00 +0.00
+0.00 +0.00 +0.00 +0.00 +0.00
reset: 0 {}
one step: 0 -0.04 False False {}


## 2. Returns and the discount

A rollout gives rewards; the return is their discounted sum. Changing $\gamma$ changes what the agent is
optimising, not just the scale of the number.

In [3]:
def rollout_return(env, steps=10, gamma=0.9):
    obs, _ = env.reset(seed=0)
    total, discount, rewards = 0.0, 1.0, []
    for _ in range(steps):
        obs, r, term, trunc, _ = env.step(0)
        rewards.append(round(float(r), 3))
        total += discount * r
        discount *= gamma
        if term or trunc:
            break
    return rewards, total

rewards, total = rollout_return(env)
print("rewards:", rewards)
print("discounted return:", round(total, 3))

rewards: [-0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04, -0.04]
discounted return: -0.261


## 3. Value iteration at three discounts

Watch the sweep count and the start-state value. A higher discount means the step cost is paid over more
steps, so the value of starting falls.

In [4]:
for gamma in (0.5, 0.9, 0.99):
    result = dp.value_iteration(mdp, gamma=gamma)
    print(f"gamma={gamma}: sweeps={result['sweeps']:2d} start value={result['V'][0]:.4f} "
          f"max value={result['V'].max():.4f}")

gamma=0.5: sweeps= 9 start value=-0.0716 max value=1.0000
gamma=0.9: sweeps= 9 start value=0.2696 max value=1.0000
gamma=0.99: sweeps= 9 start value=0.6603 max value=1.0000


## 4. Policy iteration agrees — and costs differently

Policy iteration evaluates fully between improvements, so it needs fewer improvement rounds but more work
per round. Both must converge to the same policy on a finite MDP.

In [5]:
vi = dp.value_iteration(mdp, gamma=0.9)
pi = dp.policy_iteration(mdp, gamma=0.9)
print("same policy:", bool((vi["policy"] == pi["policy"]).all()))
print("VI sweeps:", vi["sweeps"], "| PI rounds:", pi["rounds"])
print(dp.format_policy(mdp, vi["policy"]))

same policy: True
VI sweeps: 9 | PI rounds: 8
> > > > v
> > > > v
> v P > v
> > > > v
> > > > G


## 5. Monte Carlo prediction versus exact values

Monte Carlo needs no model but waits for episodes and has variance. Compare its estimate with the exact
value of the same random policy.

In [6]:
random_pi = dp.random_policy(mdp, seed=0)
exact = dp.exact_policy_value(mdp, random_pi, gamma=0.9)
print("exact start value (random policy):", round(float(exact[0]), 4))

policy_fn = lambda s: int(random_pi[s])
mc = tabular.mc_prediction(env, policy_fn, episodes=200, gamma=0.9, seed=0)
first_keys = list(mc.keys())[:3]
print("MC estimates:", {k: round(float(mc[k]), 3) for k in first_keys})

exact start value (random policy): -0.4


MC estimates: {0: -0.382}


## 6. Exercise

1. Recompute the value grid at $\gamma=0.99$ and explain the shape of the values near the goal.
2. Increase the step cost to −1.0 and describe how the greedy policy changes.
3. Estimate the random policy's value with 50 and 500 MC episodes; plot error against exact values.

## Self-assessment

- Why does higher $\gamma$ increase the number of sweeps?
- Why can policy iteration need fewer rounds than value iteration?
- Which algorithm would you use if you had samples but no model, and why?